In [1]:
import MyTools as MT
from importlib import reload
import Variable as V
import sqlite3
import pandas as pd
from datetime import datetime, timedelta, date
import Func as F

In [ ]:
conn = sqlite3.connect('BD.db') 
cursor = conn.cursor()

col_dtype = {
        'dt':'str'
    , 'stolp_den':'str'
    }

sql = """
SELECT  
substring(dt,1,10) as dt, max(deg) as deg, stolp_den 
FROM t_bazci
where 1 = 1 
group by substring(dt,1,10), stolp_den
"""

df = pd.read_sql_query(sql, conn, dtype=col_dtype)
conn.close()

In [ ]:
# Преобразование столбца dt в тип datetime
df['dt'] = pd.to_datetime(df['dt']).dt.date

# Преобразование deg в целые числа, заменяя NaN на 0 или другим значением
df['deg'] = pd.to_numeric(df['deg'], errors='coerce').fillna(0).astype(int)

# Сортировка данных по дате
df = df.sort_values(by='dt').reset_index(drop=True)

# Расчет звезды по году

In [ ]:
deg = None
star_for_year = []
gua_number = None
for index, row in df.iterrows():
    if row['deg'] == 315: # Новаый год
        gua_number = 10 - (row['dt'].year % 100 % 9 + 1)
        if gua_number == 0:
            gua_number = 1
    star_for_year.append(gua_number)

# Расчет звезды по дню

In [ ]:
current_star = None
counting = None
star_for_day = []
first_stolb = None 
deg = None
for index, row in df.iterrows():
    if row['deg'] == 270:  # Зимнее солнцестояние
        first_stolb = 0
        deg = 270
    elif row['deg'] == 90:  # Летнее солнцестояние
        current_star = 10
        first_stolb = 0
        deg = 90

    #if row['stolp_den'] <> '甲子' and first_stolb == 0: # Переходный период до первого 甲子 сохраняем счет
        
    if row['stolp_den'] == '甲子' and first_stolb == 0:
        if deg == 270:
            current_star = 0 
            counting = 1
        elif deg == 90:
            current_star = 10
            counting = -1
        
        first_stolb = 1    
        
    
    if counting == 1:
        current_star += 1
        if current_star > 9:
            current_star = 1

    if counting == -1:
        current_star -= 1
        if current_star < 1:
            current_star = 9

    star_for_day.append(current_star)

In [ ]:
conn = sqlite3.connect('BD.db') 
cursor = conn.cursor()

# Добавление новой колонки в DataFrame
df['star_for_year'] = star_for_year
df['star_for_day'] = star_for_day

# Сохранение результата в новую таблицу
df[['dt', 'deg', 'stolp_den', 'star_for_year', 'star_for_day']].to_sql('t_star_for_day', conn, if_exists='replace', index=False)
cursor.close()

# Распределение лятещих звезд по дворцам (сторонам света)

In [28]:
# Считываю из v_bazi Dt, Инь или Ян период,звезду года / месяца / дня
conn = sqlite3.connect('BD.db') 
cursor = conn.cursor()

col_dtype = {
        'dt':'str'
    , 'star_for_year':'int'
    , 'star_for_month':'int'
    , 'star_for_day':'int'
    }

sql = """
SELECT  distinct
dt,  star_for_year, star_for_month, star_for_day
FROM v_bazci
where 1 = 1 

"""

df = pd.read_sql_query(sql, conn, dtype=col_dtype)
conn.close()

In [29]:
# Прохожусь по каждой строке формируя массив  dt, флаг какая уровень (1 - год, 2 - месяц, 3 - день), дворец, звезды
fly_star = []
for index, row in df.iterrows():
    # годовая звезда
    num = row['star_for_year']
    lo_shu = [6, 7, 8, 9, 1, 2, 3, 4]
    st_num = [1, 2, 3, 4, 5, 6, 7, 8, 9]
    star = st_num[st_num.index(num) + 1:] + st_num[:st_num.index(num)]
    god = [[row['dt'], 1, a, b] for a, b in zip(lo_shu, star)]
    
    num = row['star_for_month']
    lo_shu = [6, 7, 8, 9, 1, 2, 3, 4]
    st_num = [1, 2, 3, 4, 5, 6, 7, 8, 9]
    star = st_num[st_num.index(num) + 1:] + st_num[:st_num.index(num)]
    mes = [[row['dt'], 2, a, b] for a, b in zip(lo_shu, star)]
    
    num = row['star_for_day']
    lo_shu = [6, 7, 8, 9, 1, 2, 3, 4]
    st_num = [1, 2, 3, 4, 5, 6, 7, 8, 9]
    star = st_num[st_num.index(num) + 1:] + st_num[:st_num.index(num)]
    den = [[row['dt'], 3, a, b] for a, b in zip(lo_shu, star)]
    
    fly_star = fly_star + god
    fly_star = fly_star + mes
    fly_star = fly_star + den
    pass

In [30]:
# Преобразую массив в DataFrame
df_fly = pd.DataFrame(fly_star, columns = ['dt', 'level', 'palace', 'star'])
df_fly['dt'] = pd.to_datetime(df_fly['dt']).dt.date

In [31]:
# Преобразую плоский вид таблицы в перекресный: строки - дата, дворец; колонки уровни god, mes, den - на пересечении звезды
df_pivot = pd.pivot_table (df_fly, index = ['dt', 'palace'],
                                   columns = ['level'],
                                   values = 'star',
                                   aggfunc = 'sum'
                          )
df_pivot.reset_index(inplace=True)
# Переименовываем колонки
df_pivot = df_pivot.rename(columns={1: 'year_star', 2: 'month_star', 3: 'day_star'})

In [39]:
df_pivot.head(10)

level,dt,palace,year_star,month_star,day_star
0,2025-01-13,1,8,8,6
1,2025-01-13,2,9,9,7
2,2025-01-13,3,1,1,8
3,2025-01-13,4,2,2,9
4,2025-01-13,6,4,4,2
5,2025-01-13,7,5,5,3
6,2025-01-13,8,6,6,4
7,2025-01-13,9,7,7,5
8,2025-01-14,1,8,8,7
9,2025-01-14,2,9,9,8


### Летящие звезды часа

In [33]:
# Считываю из v_bazi Dt, Инь или Ян период,звезду года / месяца / дня
conn = sqlite3.connect('BD.db') 
cursor = conn.cursor()

col_dtype = {
        'dt':'str'
    , 'name_hour':'str'
    , 'star_for_hour':'int'
    }

sql = """
SELECT  distinct
dt,  name_hour, star_for_hour
FROM v_bazci
where 1 = 1 
and star_for_hour is not null

"""

df_hour = pd.read_sql_query(sql, conn, dtype=col_dtype)
conn.close()

In [14]:
fly_star_hour = []
for index, row in df_hour.iterrows():
    num = row['star_for_hour']
    lo_shu = [6, 7, 8, 9, 1, 2, 3, 4]
    st_num = [1, 2, 3, 4, 5, 6, 7, 8, 9]
    star = st_num[st_num.index(num) + 1:] + st_num[:st_num.index(num)]
    chas = [[row['dt'],row['name_hour'] , 4, a, b] for a, b in zip(lo_shu, star)]
    fly_star_hour = fly_star_hour + chas

In [34]:
fly_star_hour[0]

['2025-01-13', 'Ранняя крыса', 4, 6, 2]

In [35]:
# Преобразую массив в DataFrame
df_fly_hour = pd.DataFrame(fly_star_hour, columns = ['dt', 'name_hour' ,'level', 'palace', 'star'])
df_fly_hour['dt'] = pd.to_datetime(df_fly_hour['dt']).dt.date

In [37]:
# Преобразую плоский вид таблицы в перекресный: строки - дата, дворец; колонки уровни god, mes, den - на пересечении звезды
df_pivot_hour = pd.pivot_table (df_fly_hour, index = ['dt','name_hour' ,'palace'],
                                   columns = ['level'],
                                   values = 'star',
                                   aggfunc = 'sum'
                          )
df_pivot_hour.reset_index(inplace=True)


In [40]:
# Переименовываем колонки
df_pivot_hour = df_pivot_hour.rename(columns={1: 'dt', 2: 'name_hour', 3: 'palace', 4:'hour_star'})

In [44]:
# Объединяем звезды года / месяца / дня и часа
df_total = pd.merge(df_pivot_hour, df_pivot, on = ['dt', 'palace'], how = 'left')
df_total = df_total[['dt', 'name_hour', 'palace', 'year_star', 'month_star', 'day_star', 'hour_star']]

In [48]:
conn = sqlite3.connect('BD.db') 
cursor = conn.cursor()

# Записываю DataFrame в БД как t_fly_star_in_palace
df_total.to_sql('t_fly_star_in_palace', conn, if_exists='replace', index=False)

187616